# FutureCV — E5-small + bộ phân loại + nhận xét theo mẫu

Pipeline CPU: JD thô → tự tách yêu cầu/gán weight → E5 đóng băng → bộ phân loại
nhỏ → tính điểm/nhận xét bằng code. Nhà tuyển dụng không nhập weight.
Chọn classifier bằng validation.

Upload `futurecv_e5_bundle.zip` lên
`MyDrive/FutureCV-ML/lightweight/futurecv_e5_bundle.zip` trước khi chạy.
ZIP đã có code và dataset tổng hợp. Xem README trong ZIP để deploy VPS.

## 1. Cài thư viện
Chạy một lần mỗi runtime mới. Giữ PyTorch/CUDA của Colab.
Nếu Colab yêu cầu restart, restart rồi chạy từ mục 2.

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "numpy>=1.26,<3", "transformers==4.57.1", "scikit-learn==1.7.2",
    "fastapi==0.119.1", "uvicorn==0.38.0"])
print("Cài xong. Nếu cần restart theo thông báo Colab, restart rồi chạy mục 2.")

## 2. Mount Drive và giải nén bundle
Giải nén code/data; model được lưu riêng trên Drive.

In [ ]:
from pathlib import Path
from zipfile import ZipFile
from google.colab import drive
drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/FutureCV-ML")
BUNDLE = DRIVE_ROOT / "lightweight/futurecv_e5_bundle.zip"
assert BUNDLE.is_file(), f"Chưa có file: {BUNDLE}"
with ZipFile(BUNDLE) as archive:
    for name in archive.namelist():
        target = (Path("/content") / name).resolve()
        if not target.is_relative_to(Path("/content/futurecv_e5")):
            raise ValueError(f"Đường dẫn ZIP không hợp lệ: {name}")
    archive.extractall("/content")
CODE_DIR = Path("/content/futurecv_e5")
DATA_DIR = CODE_DIR / "data"
print("Code:", CODE_DIR, "Dataset:", DATA_DIR)

## 3. Cấu hình và kiểm tra dữ liệu
Lần đầu để smoke. Khi smoke xong, đổi RUN_MODE thành full rồi chạy lại mục 3 → 4.
Nếu output cũ đã tồn tại, đổi RUN_NAME để giữ nguyên thí nghiệm cũ.
Không random split lại 2.800 requirements.

In [ ]:
import sys, torch
RUN_MODE = "smoke"  # đổi thành "full" sau smoke
RUN_NAME = "e5-small-classifier-v1"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
assert RUN_MODE in ("smoke", "full")
assert RUN_NAME and Path(RUN_NAME).name == RUN_NAME and RUN_NAME not in (".", "..")
RUN_DIR = DRIVE_ROOT / RUN_NAME / RUN_MODE
sys.path.insert(0, str(CODE_DIR))
from pipeline import load_splits, target_labels
splits, hashes = load_splits(DATA_DIR)
print({name: {"pairs": len(rows), "requirements": len(target_labels(rows))} for name, rows in splits.items()})
print("Train device:", DEVICE, "Output:", RUN_DIR)

## 4. Train classifier
E5 không được cập nhật trọng số. Code train 3 Logistic Regression và 2 MLP64,
chọn theo macro-F1 validation. Smoke dùng 40/20 cặp, full dùng 700/100 cặp.
Encoder đóng băng, chỉ huấn luyện bộ phân loại.

In [ ]:
import subprocess, sys
command = [sys.executable, "-u", str(CODE_DIR / "train.py"),
    "--data-dir", str(DATA_DIR), "--output-dir", str(RUN_DIR),
    "--device", DEVICE, "--threads", "2"]
if RUN_MODE == "smoke":
    command.append("--smoke")
subprocess.run(command, check=True)
print("Bước tiếp theo: đổi full và chạy mục 3 → 4" if RUN_MODE == "smoke" else "Full xong. Xem validation trước khi chạy mục 5.")

## 5. Đánh giá test trên CPU — chỉ sau full
Chốt cấu hình bằng validation trước. Không tối ưu lại theo test.
Giữ nguyên các file kết quả; không ghi đè đánh giá cũ.
Nếu runtime mới nhưng đã có full artifact: chạy mục 1 → 3 với RUN_MODE full,
bỏ qua mục 4 rồi chạy ở đây.

In [ ]:
import subprocess, sys
assert RUN_MODE == "full", "Không đánh giá test với smoke"
subprocess.run([sys.executable, "-u", str(CODE_DIR / "evaluate.py"),
    "--data-dir", str(DATA_DIR), "--artifact-dir", str(RUN_DIR),
    "--device", "cpu", "--threads", "2"], check=True)

## 6. Thử CV mới
Sửa example_raw_input.json: chỉ cần cv_text, jd_text và job_title nếu có.
Parser tự tạo requirements/weight và lưu phiên bản trong kết quả.
Nếu JD không nằm trong phạm vi parser, trả needs_review và không tính điểm tổng.
Input chuẩn hóa example_input.json vẫn được giữ để tái lập benchmark cũ.
Nhận xét theo mẫu và ngữ cảnh truy xuất cần được đối chiếu, không phải bằng
chứng đã được người xác minh. Điểm được tính cố định từ nhãn dự đoán.

In [ ]:
import subprocess, sys
assert RUN_MODE == "full"
completed = subprocess.run([sys.executable, "-u", str(CODE_DIR / "predict.py"),
    "--artifact-dir", str(RUN_DIR), "--input", str(CODE_DIR / "example_raw_input.json")],
    capture_output=True, text=True, encoding="utf-8")
print(completed.stdout)
if completed.stderr:
    print(completed.stderr)
completed.check_returncode()

### 6b. Chuẩn bị JD riêng khi đăng tin — không cần nạp E5
Cell này nhận JD nguyên văn và tự gán weight. Profile ready có thể được backend
lưu cùng job và gửi lại qua trường job_profile khi chấm từng CV.
Xem END_TO_END.md để biết phạm vi và các câu bị đánh dấu cần kiểm tra.

In [ ]:
import json, sys, importlib
from pathlib import Path
CODE_DIR = Path(globals().get("CODE_DIR", "/content/futurecv_e5"))
if not (CODE_DIR / "jd_parser.py").is_file():
    raise FileNotFoundError(f"Thiếu {CODE_DIR / 'jd_parser.py'}. Chạy lại mục 2 để mount Drive/giải nén ZIP v1, rồi chạy lại cell này. Không cần train lại.")
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
importlib.invalidate_caches()
from jd_parser import prepare_jd
raw_example = json.loads((CODE_DIR / "example_raw_input.json").read_text(encoding="utf-8"))
job_profile = prepare_jd(raw_example["jd_text"], raw_example["job_title"])
print("Trạng thái:", job_profile["status"])
print(json.dumps(job_profile["requirements"], ensure_ascii=False, indent=2))
print("Chưa xử lý:", job_profile["unresolved"])

## 7. Đóng gói để deploy CPU
ZIP chỉ chứa code và artifact full, không đóng gói CV đầu vào hoặc prediction
test. ZIP lưu trên Drive để tải về. Upload lên VPS và làm theo README.

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
assert RUN_MODE == "full" and (RUN_DIR / "manifest.json").is_file()
DEPLOY_ZIP = RUN_DIR.parent / f"{RUN_NAME}-deploy.zip"
if DEPLOY_ZIP.exists():
    raise FileExistsError(f"Đã có {DEPLOY_ZIP}; giữ lại kết quả hoặc chọn tên khác")
with ZipFile(DEPLOY_ZIP, "w", ZIP_DEFLATED) as archive:
    for name in ("pipeline.py", "jd_parser.py", "prepare_jd.py", "predict.py", "evaluate.py", "server.py", "requirements.txt", "example_input.json", "example_raw_input.json", "example_raw_jd.txt", "README.md", "END_TO_END.md", "THIRD_PARTY.md"):
        archive.write(CODE_DIR / name, name)
    for name in ("manifest.json", "classifier.npz", "THIRD_PARTY.md"):
        archive.write(RUN_DIR / name, "artifact/" + name)
    for path in (RUN_DIR / "encoder").rglob("*"):
        if path.is_file():
            archive.write(path, "artifact/encoder/" + path.relative_to(RUN_DIR / "encoder").as_posix())
print("ZIP deploy:", DEPLOY_ZIP)
print("Đọc README mục Deploy VPS. Dùng một worker, không cần GPU.")